# BIO-FMI on SARS-CoV-2: what `l` buys, what it costs, and one bug

**Dataset:** COVID-294 — 294 SARS-CoV-2 genomes, 34,288 alignment columns,
`ctx_avg` 18.85, 294 paths.
**Anchor:** `|P| = 120`, 200 source-aware patterns, seed 7.

This notebook covers three things:

1. **The trade-off (E5).** What raising the context length `l` buys at query
   time and costs in index size, for both merge modes.
2. **Precision.** `l` turns out to be a precision knob, not only a space/time
   knob — a result nobody predicted.
3. **A correctness failure (B4).** The reported occurrence counts are wrong at
   *every* `l`, and this notebook shows exactly why.

### Why `l` and `|P|` are what they are

BIO-FMI splits a pattern into `|P| / (l+1)` chunks and rejects any pattern that
does not divide evenly. So a sweep over `l` values whose `l+1` share no common
multiple cannot be plotted against a fixed pattern length at all. Every `l`
below satisfies `(l+1) | 120`:

```
l ∈ {3, 5, 9, 11, 14, 19, 29, 39, 59}
```

`|P| = 120` is both highly composite and a plausible read length.

### Reproducing the inputs

Everything the sweep needs is declared in `experiments/specs/merge_mode.yaml`
and run by the xbench harness — merge, index build and query, all nine `l`, both
merge modes:

```bash
./experiments/run.sh merge_mode

./experiments/occurrence_oracle.py \
    --msa ~/Data/covid/raw/all_sequences.msa \
    --patterns <run>/patterns/covid294.real.pattern_len120.txt \
    --out ~/Data/covid/work/covid294/oracle/real.tsv
```

Per-pattern locate counts are archived per run under
`~/Data/experiments/biofmi/runs/merge_mode/<stamp>/raw/query/<cell>/real/rep1.stdout.log`, one
`<pattern>\t<count>` per line; the cells below read the copies collected into
`~/Data/covid/work/covid294/locate/`.

> **Provenance of the numbers below.** They come from edsparser `fbfbdec`, whose
> `genpatterns` did not yet deduplicate — so the pattern set is 200 lines holding
> 198 distinct patterns, and every comparison below is careful to use the same
> 198 on both sides. Since `e79dd33` patterns are distinct by default, so a fresh
> run draws a *different* set and different absolute counts. The ceiling argument
> in §3 holds for any pattern set; the specific totals here belong to this one.

In [ ]:
from pathlib import Path
import collections
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# The committed reference bundle: small, versioned, and what
# specs/acceptance_covid294.py checks a fresh run against.
REPO    = Path.home() / "Documents/uni_projects/biofmi"
RESULTS = REPO / "experiments/results/covid294"
WORK    = Path.home() / "Data/covid/work/covid294"

# Palette: two categorical slots (validated for CVD separation), status red for
# violations, and recessive ink/grid tokens so the data stays the loudest thing
# on the page.
SERIES   = {"linear": "#2a78d6", "cartesian": "#eb6834"}
CRITICAL = "#d03b3b"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.family": "sans-serif", "font.size": 10,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "semibold",
    "axes.titlelocation": "left", "axes.titlepad": 10,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "grid.color": GRID, "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.labelcolor": INK2,
    "figure.dpi": 110,
})

def style(ax, ylabel=None, xlabel=None):
    """Recessive chrome: horizontal grid behind the marks, no box."""
    ax.grid(True, axis="y", zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if ylabel: ax.set_ylabel(ylabel)
    if xlabel: ax.set_xlabel(xlabel)

res = pd.read_csv(RESULTS / "results.csv")
qry = pd.read_csv(RESULTS / "queries.csv")
L_VALUES = sorted(int(v) for v in res.l.unique())
print(f"{len(L_VALUES)} l values: {L_VALUES}")

## 1. Build cost and feasibility

Both modes merge the same EDS. The only difference is one flag: LINEAR passes
the source file, so the merge keeps only combinations some genome carries;
CARTESIAN omits it and keeps every combination of adjacent alternatives.

That single flag decides how far the sweep can go.

In [ ]:
summary = pd.DataFrame({
    "linear .leds MB":    (res[res["mode"]=="linear"].set_index("l").leds_bytes / 1e6).round(2),
    "cartesian .leds MB": (res[res["mode"]=="cartesian"].set_index("l").leds_bytes / 1e6).round(2),
    "cartesian status":    res[res["mode"]=="cartesian"].set_index("l").merge_status,
})
summary["blowup ×"] = (summary["cartesian .leds MB"] / summary["linear .leds MB"]).round(1)
summary.loc[summary["cartesian status"] != "ok", ["cartesian .leds MB", "blowup ×"]] = None
summary

Cartesian is larger at every `l` and the gap compounds — 1.3× at `l=3`, **27.5×
at `l=14`** — then it stops being buildable at all. Linear reaches `l=59` (and
`l=399` in an earlier probe) at under 4 MB.

**Phasing widens the usable `l` range by more than 4×**, which matters because
`l` is the knob that buys query speed.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
real = qry[qry.pattern_set == "real"]

for mode, c in SERIES.items():
    d = real[real["mode"] == mode].sort_values("l")
    ax1.plot(d.l, d.per_pattern_ms, "-o", color=c, lw=2, ms=6, label=mode, zorder=3)
    ok = res[(res["mode"] == mode) & (res.merge_status == "ok")].sort_values("l")
    ax2.plot(ok.l, ok.index_bytes / 1e6, "-o", color=c, lw=2, ms=6, label=mode, zorder=3)

ax1.set_yscale("log")
ax1.set_title("Query cost falls sharply with l")
style(ax1, "ms per pattern (log)", "context length l")
ax1.legend(loc="upper right")

ax2.set_yscale("log")
ax2.set_title("Index size is the price")
style(ax2, "index size, MB (log)", "context length l")
ax2.annotate("cartesian OOM\nfrom l = 19", xy=(14, 26.4), xytext=(24, 9),
             color=INK2, fontsize=9,
             arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))
plt.show()

**The headline.** Linear query cost drops from 87.9 ms/pattern at `l=3` to
0.06 ms at `l=9` — three orders of magnitude — while the index grows only
0.85 → 0.91 MB. The mechanism is plain: `|P|/(l+1)` chunks, so 30 chunks at
`l=3` against 2 at `l=59`.

The curve flattens past `l≈9`: once the chunk count is small, chunking is no
longer what dominates. Beyond that point extra `l` buys index size, not speed —
so the interesting region for this dataset is `l ∈ [9, 19]`, not the top of the
range.

## 2. `l` is also a precision knob

This was not anticipated. The **decoy set** is patterns generated *ignoring*
sources — strings assembled from alternatives that no single genome carries —
filtered to those the linear `l=59` index rejects. A representation that admits
them is inventing sequence.

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.0))
dec = qry[qry.pattern_set == "decoy"]
for mode, c in SERIES.items():
    d = dec[dec["mode"] == mode].sort_values("l")
    ax.plot(d.l, d.matched, "-o", color=c, lw=2, ms=6, label=mode, zorder=3)
ax.set_title("Larger l rejects more sequence no genome carries")
style(ax, "decoys admitted (of 200)", "context length l")
ax.legend(loc="upper right")
ax.annotate("0 at l = 59", xy=(59, 0), xytext=(44, 22), color=INK2, fontsize=9,
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))
plt.show()

Linear admits 119/200 decoys at `l=3` and 0 at `l=59`, falling monotonically.
Larger `l` merges more, so more of the source constraint is materialised into
the strings themselves and the index enforces more haplotype consistency.

Cartesian sits flat at **149/200 at every `l` it can build**. It has no source
information to enforce, so merging more does not make it more faithful.

*(`l=59` is 0 by construction — it defines the set. The eight points below it
are free measurements.)*

Both modes find 200/200 real patterns and 0/200 negative controls at every `l`,
so this is precision, not a recall or thresholding artefact.

## 3. B4 — the occurrence counts are wrong

The reported occurrence count for one fixed pattern set swings
**1,950 → 65,723 → 31,388** as `l` goes 3 → 39 → 59. Whether a pattern occurs
in a genome, and at how many offsets, is a property of the genomes alone. It
**cannot** depend on `l`.

So either the counts measure something other than genome occurrences, or they
are wrong — and nothing in the repo could tell which. The only existing ground
truth (`brute_force_locate` in `test_locate_correctness.cpp`) expands the
*cartesian* language, which is exponential and is also the wrong semantics for a
LINEAR index; it runs only at `l=3` and `l=4` on hand-written strings.

### The oracle

`experiments/occurrence_oracle.py` goes back to the alignment the whole pipeline
was built from and materialises each genome by dropping its gap columns. It
shares no code with edsparser or BIO-FMI *on purpose* — an oracle that agrees
with the index because both have the same bug is worthless.

```bash
./experiments/occurrence_oracle.py \
    --msa ~/Data/covid/raw/all_sequences.msa \
    --patterns ~/Data/covid/work/covid294/patterns/real.txt \
    --out ~/Data/covid/work/covid294/oracle/real.tsv
```

### The ceiling

Each (genome, offset) occurrence corresponds to exactly one
(position, change-combination). Two genomes agreeing on every choice the pattern
spans collapse onto the same one. Therefore

$$\text{distinct entries} \;\le\; \text{(genome, offset) occurrences}$$

and the right-hand side does not depend on `l`.

In [ ]:
oracle = pd.read_csv(WORK / "oracle" / "real.tsv", sep="\t")
counts = {l: pd.read_csv(WORK / "locate" / f"counts_l{l}.tsv", sep="\t",
                         names=["pattern", "entries"])
          for l in L_VALUES if (WORK / "locate" / f"counts_l{l}.tsv").exists()}

ceiling = oracle.drop_duplicates("pattern").total_occurrences.sum()
matched = (oracle.total_occurrences > 0).sum()

print(f"patterns              {len(oracle)} ({oracle.pattern.nunique()} distinct)")
print(f"matched in the panel  {matched}/{len(oracle)}")
print(f"ceiling               {ceiling:,} occurrences  (l-invariant)")

**200/200 patterns are real substrings of real genomes.** That is worth pausing
on: it independently validates edsparser's source-aware `genpatterns` against a
completely separate implementation.

*(`real.txt` has 198 distinct patterns out of 200 lines — `genpatterns` does not
deduplicate. Harmless for timings; it means "200 patterns" is 198 distinct
queries.)*

In [ ]:
# Dedupe on pattern so the totals and the ceiling are computed over the same
# 198 distinct patterns (real.txt has two repeats — genpatterns does not
# deduplicate). Comparing a 200-row sum against a 198-pattern ceiling would
# be apples to oranges.
tot   = [counts[l].drop_duplicates("pattern").entries.sum() for l in L_VALUES]
overs = []
for l in L_VALUES:
    m = counts[l].drop_duplicates("pattern").merge(oracle.drop_duplicates("pattern"), on="pattern")
    overs.append(int((m.entries > m.total_occurrences).sum()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
x = range(len(L_VALUES))

ax1.bar(x, tot, width=0.62, zorder=3,
        color=[CRITICAL if t > ceiling else SERIES["linear"] for t in tot])
ax1.axhline(ceiling, color=INK2, lw=1.4, ls=(0, (5, 3)), zorder=4)
ax1.text(-0.4, ceiling * 1.06, f"ceiling {ceiling:,}", ha="left",
         color=INK2, fontsize=9)
ax1.set_xticks(list(x), [str(l) for l in L_VALUES])
ax1.set_title("Entries reported vs what the genomes allow")
style(ax1, "entries returned", "context length l")
ax1.annotate(f"{tot[7]:,}", xy=(7, tot[7]), xytext=(0, 6), textcoords="offset points",
             ha="center", color=CRITICAL, fontsize=9, fontweight="bold")

ax2.bar(x, overs, color=CRITICAL, width=0.62, zorder=3)
ax2.set_xticks(list(x), [str(l) for l in L_VALUES])
ax2.set_title("Patterns over their own ceiling — at every l")
style(ax2, "patterns exceeding truth", "context length l")
for i, v in enumerate(overs):
    ax2.annotate(str(v), xy=(i, v), xytext=(0, 4), textcoords="offset points",
                 ha="center", color=INK2, fontsize=9)
plt.show()

Two readings, and the second is the important one:

- **Aggregate** (left): only `l=39` breaks the ceiling outright, at 65,668
  against 48,174. Everywhere else the total sits *below* it, because genomes
  sharing choices legitimately collapse onto one entry.
- **Per pattern** (right): the ceiling is violated at **every single `l`**.
  It was never a large-`l` phenomenon — at `l=3` it is just small enough to look
  like noise.

`compare_locate_oracle.py` exits non-zero on any breach, so this can gate a run
rather than being a one-off investigation.

## 4. The mechanism

The worst case is pattern 138 at `l=39`: **57,340 entries against 285 true
occurrences** — once in each of 285 genomes. A 201× over-report, and it alone
accounts for 57,055 of the 57,530 excess entries at that `l`.

All 57,340 entries are **distinct**, so nothing is being double-counted, and
they span only **15 positions**. Breaking them down per position gives the
answer.

In [ ]:
by_pos = collections.defaultdict(list)
for line in (WORK / "locate" / "pattern138_l39_full.txt").read_text().splitlines():
    if not line[:1].isdigit():
        continue
    pos, rest = line.split("[", 1)
    by_pos[int(pos)].append(tuple(int(v) for v in rest.strip(" ]").split()))

mech = pd.DataFrame(
    [(pos,
      len(chs),
      len({c[0] for c in chs}),
      len({c[1] for c in chs if len(c) > 1}))
     for pos, chs in sorted(by_pos.items())],
    columns=["position", "entries", "distinct_c1", "distinct_c2"])
mech["cross"] = mech.distinct_c1 * mech.distinct_c2
mech["exact"] = mech.entries == mech.cross
mech

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.2))
lo, hi = mech.cross.min() * 0.6, mech.cross.max() * 1.7
ax.plot([lo, hi], [lo, hi], color=MUTED, lw=1.2, ls=(0, (5, 3)), zorder=2,
        label="entries = |c₁| × |c₂|")
ax.scatter(mech.cross, mech.entries, s=64, color=SERIES["linear"],
           edgecolor=SURFACE, linewidth=1.4, zorder=3, label="one T₀ position")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_title("Every position sits exactly on the cross product")
style(ax, "entries returned (log)", "|c₁| × |c₂| (log)")
ax.legend(loc="upper left")
plt.show()

print(f"exact matches: {mech.exact.sum()} of {len(mech)} positions")
print("(several positions coincide at (244, 244), so fewer than 15 dots are visible)")

`entries = |c₁| × |c₂|` **exactly, at all 15 positions.**

`locate` pairs every alternative of one degenerate symbol with every alternative
of the next, without intersecting their source sets. The l-EDS prunes
combinations *within* a merged symbol — that is what LINEAR merging does — but a
pattern spanning *two* merged symbols gets the full cross product, including
walks no genome carries.

This is the same mechanism `experiment_design.md` §5 guessed at for the decoy
result: *"at small `l` a chunked search can stitch together a walk no path
carries."* It is now demonstrated — and it inflates counts for genuine patterns
too, not only decoys.

## 5. What stands, and what does not

**Unaffected — recall.** 200/200 real patterns match at every `l`; 0/200
negative controls match. The oracle independently confirms all 200 are real
substrings of real genomes.

**Unaffected — size, feasibility, timing.** None depends on how many entries
come back. Worth checking the obvious worry — that the speedup was really "fewer
spurious entries at high `l`" — and it is not: `l=3` has the *fewest* entries
(1,946) and the *slowest* query (87.9 ms/pattern), so the speedup tracks chunk
count, as claimed.

**Invalid — the occurrence and entry columns.** They are not counts of anything
real and need regenerating after a fix.

**The decoy result survives but changes meaning.** The *shape* is real and now
explained: at small `l` there is more cross-chunk stitching, so more decoys get
through. It is a measurement of this same bug's severity as a function of `l`,
which is arguably more interesting than the original framing.

### The fix

In `locate`, when combining chunk matches across degenerate symbols, intersect
the source sets and drop empty intersections — the same pruning
`compute_merge_metadata()` already does on the edsparser side. The open question
is whether the index still *retains* the source information needed to do that,
or whether it has to be added to the on-disk format.

### Next

- Fix the intersection, then re-run §3 and §4; both should collapse to "at or
  below ceiling" everywhere.
- Re-measure E1 with the corrected counts and the ≥10-repetition protocol
  (median + p95, discarded warmup).
- E3 — stratify patterns by pure-reference / one degenerate symbol / ≥2 /
  negative controls, which is the experiment that shows what the dual-index
  design costs when a match spans variation.